# Tutorial: Simulating ancient navigation with Pytheas Simulator

> Author: Matteo Tomasini
>
> Updated: 12.10.2026

IMPORTANT NOTE: this tutorial is not as of yet tested. I will do it in the next couple of weeks.

In this tutorial, I will go through the steps that are needed to get Python Simulator to work on your laptop, as well as running a small simulation example. In this Python Notebook I will go through the important steps, but there is also a script to run once you have downloaded the data and set up the simulator. 

The aim of the tutorial is for you to familiarize yourself with Pytheas Simulator, by simulating a crossing and a coastal travel in the Kattegat strait. The tutorial will also show you how to download data for one year (to illustrate how the data folders are structured).

Let's get going! 

## The data

First, we need to download the environmental data necessary for Pytheas to function: currents, winds, and waves. The data need to be structured in a specific format. From your data folder, the structure is:

```
.
├── currents
│   └── 2025
│       └── month_1.nc
|       └── month_2.nc
|       └── ...
|       └── month_12.nc
├── waves
│   └── 2025
│       └── month_1.nc
|       └── month_2.nc
|       └── ...
|       └── month_12.nc
└── winds
        └── 2025
            └── month_1.nc
            └── month_2.nc
            └── ...
            └── month_12.nc
```

If you have downloaded this tutorial and the corresponding folders from https://github.com/mtomasini/PytheasSimulator/, the data folder is `./data/environmental_data/`.

### Copernicus data stores

To download the data from the Copernicus Data Store, you first need to obtain a login for the [ECMWF Climate Data Store](https://cds.climate.copernicus.eu/) and the [Copernicus Marine Data Store](https://data.marine.copernicus.eu/) -- in both cases, click on "Login" and follow the indications. Once you have obtained these, you can go through with the rest of the tutorial.

Some additional set-up is needed for the provided script to work. The process is different for different operating systems, please refer to the official documentation to find your set-up:

- ECMWF Data Store: https://cds.climate.copernicus.eu/how-to-api
- Copernicus Marine Data Store: https://help.marine.copernicus.eu/en/articles/7949409-copernicus-marine-toolbox-introduction

### Winds download

The winds are downloaded from the [CERRA sub-daily regional reanalysis data for Europe on single levels from 1984 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-cerra-single-levels?tab=overview); in the specific the script downloads the variables `10m wind direction` and `10m wind speed`. Be sure to have set up your profile and your API, and then run:

```python
import cdsapi

months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

for month in months:

    dataset = "reanalysis-cerra-single-levels"
    request = {
        "variable": [
            "10m_wind_direction",
            "10m_wind_speed"
        ],
        "level_type": "surface_or_atmosphere",
        "data_type": ["reanalysis"],
        "product_type": "analysis",
        "year": ["1995"],
        "month": [
            f"{month}"
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "time": [
            "00:00", "06:00",
            "12:00", "18:00"
        ],
        "data_format": "netcdf"
    }

    client = cdsapi.Client()
    target = f"./data/environmental_data/winds/2025/month_{month}.nc"
    client.retrieve(dataset, request, target).download()

```

ATTENTION! Do not run this code in this tutorial. It will send a request to CDS to download your data, but will put you in a queue, especially in times of high traffic. This may take a while to run! The same code can be found in the script `./data/import_winds.py`.

### Currents and waves

Currents data and waves data are downloaded from the same source. Here too, refer to the documentation to set-up your login and your API requests. For the Arctic, the datasets we are interested in are the [Arctic Ocean Physics Reanalysis](https://data.marine.copernicus.eu/product/ARCTIC_MULTIYEAR_PHY_002_003/download) (`cmems_mod_arc_phy_my_topaz4_P1D-m`) for the currents, and the waves dataset is the [Arctic Ocean Waves Hindcast](https://data.marine.copernicus.eu/product/ARCTIC_MULTIYEAR_WAV_002_013/download) (`cmems_mod_arc_wav_my_3km-climatology_P1M-m`).


Then, you can run the following script making use of the Copernicus Marine Toolbox. This will ask for your login credentials before it proceeds to downloading your two datasets.

```python
import copernicusmarine

def monthly_import(dataset, type_of_dataset):
    months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

    if type_of_dataset == "currents":
        # ATTENTION! there are datasets out there that call currents in the x and y component "uo" and "vo" instead!
        variables == ["vxo", "vyo"]
    elif type_of_dataset == "waves":
        variables == ["VHM0"]

    for month in months:
        last_day = "31" 
        if month in ["04", "06", "09", "11"]: 
            last_day = "30"
        elif month == "02":
            # here we are downloading for 2025, but in case of a leap year this should be "29"!
            last_day = "28"
        copernicusmarine.subset(
            dataset_id=dataset,
            variables=variables,
            minimum_longitude=-29.720039021480375,
            maximum_longitude=10.480836622997344,
            minimum_latitude=55.8011102398125,
            maximum_latitude=65.84004107063826,
            start_datetime=f"1995-{month}-01T00:00:00",
            end_datetime=f"1995-{month}-{last_day}T23:59:59",
            minimum_depth=0,
            maximum_depth=0,
            output_filename = f"month_{month}.nc",
            output_directory = f"./data/environmental_data/{type_of_dataset}/"
    )

monthly_import("cmems_mod_arc_phy_my_topaz4_P1D-m", "currents")
monthly_import("cmems_mod_arc_wav_my_3km-climatology_P1M-m", "waves")
```
Again, this is exactly the code you find in the script `./data/import_currents-waves.py`.

### Pre-shipped dataset

For confort of usage, I include in the data shipped with this tutorial a month of data for the Baltic for 1996. The wind data comes from the same dataset specified above (CERRA contains data for the whole of Europe, including part of the Arctic and the Mediterranean Sea); the currents come from the dataset [`cmemes_mod_bal_phy_my_P1D-m`](https://data.marine.copernicus.eu/product/BALTICSEA_MULTIYEAR_PHY_003_011/download); the waves from the dataset [`cmems_mod_bal_wav_my_PT1H-i`](https://data.marine.copernicus.eu/product/BALTICSEA_MULTIYEAR_WAV_003_015/download).

## Simulation

Let us simulate two voyages: one is a direct crossing between Gothenburg and the northernmost tip of Denmark, Skagen. The second voyage will be a coastal voyage from Gothenburg northward to Tanumshede. Let's start by defining the locations and some paths. This block contains a lot of set-up that is not necessary to explain in detail, but fundamentally, we create the necessary boiletplate to run the simulation, find the outputs, and create the right outputs. These are generic to both simulations that we are going to test:

In [ ]:
import gc
import pandas as pd
from pathlib import Path
import pytheas_simulator

# used only for the saving the output under a recognizable name
experiment_name = "tutorial_experiment"

speed_polar_diagram = pd.read_csv("./polar_diagrams/hjortspring_speeds_16pad_3000kg_44cad_75oars.txt", sep=",", index_col=0)
leeway_polar_diagram = pd.read_csv("./polar_diagrams/hjortspring_leeway_empty_diagram.csv", sep=",", index_col=0)

WIND_DATA = "./data/winds/"
CURRENT_DATA = "./data/currents/"
WAVES_DATA = "./data/waves/"

locations = {
    "GOTHENBURG" : [57.66, 11.72],
    "SKAGEN" : [57.72, 10.61],
    "TANUM" : [58.70, 11.16]
}

# this snippet creates a folder if it does not exist with the name of the experiment chosen before
Path(f"./simulation_outpu/Outputs/{output}").mkdir(parents=True, exist_ok=True)



### Crossing voyage

Now we start defining the parameters of the voyage. For our crossing between GOTHENBURG and SKAGEN, we define the following parameters:

In [ ]:

# maximal duration of each voyage: 240 hours is 10 days
max_duration_in_hours = 240 
date_of_departure = "1996-07-14"
final_day_of_navigation = "1996-07-24"
departure_site = locations["GOTHENBURG"]
landing_site = locations["SKAGEN"]
bounding_box = pytheas_simulator.utilities.bounding_box_from_points(departure_site, landing_site, buffer=4)
timestep_in_minutes = 15


It is important to note that the `departure_site` is not on the water, which would leave our boat beached. To avoid this, we run a small utility function to find the closest point on the water to the departure site: 

In [ ]:
departure_site_on_water = travel_map.find_closest_water(departure_site)

Then, we need to create the map of the voyage. The `Map` object is the "interface" containing the environmental data and determining thus how strong winds, currents and waves are at each moment in time. It is defined starting from the bounding box (itself depending on the departure and landing sites), as well as the total temporal duration of the voyage.

In [ ]:
travel_map = pytheas_simulator.map.Map(bounding_box, earliest_time=date_of_departure, latest_time=final_day_of_navigation, 
                                       wind_data_path = WIND_DATA, current_data_path=CURRENT_DATA, waves_data_path=WAVES_DATA)

Second, we define the boat instance, using the class `Boat`. Since the package is shipped with a reconstructed polar diagram for the [Hjortspring Boat](https://en.wikipedia.org/wiki/Hjortspring_boat), let's call our boat "Hjortspring". 

In [ ]:
hjortspring = pytheas_simulator.boat.Boat('Hjortspring', latitude = departure_site_on_water[0], longitude = departure_site_on_water[1],
                                          target = landing_site, speed_polar_diagram = speed_polar_diagram, 
                                          leeway_polar_diagram=leeway_polar_diagram)

NameError: name 'pytheas_simulator' is not defined

Finally, it is time to create our travel. The class `Travel` puts together the map and the boat, and makes them interact. It also acts as a logbook where different things are recorded, such as encountered environmental conditions, trajectory, and bearings. 
- The parameter `tolerance_distance` expresses at which distance from the target point the boat is considered to be at destination. This is to avoid lengthy searches for the perfect landing spot which can increase travel time in our "blind" boat system. 
- The parameter `twilights_of_stops` determines at what time the vessel starts navigating in the morning: `'civil'` here stands for _civil twilight_ which is defined as when the Sun lies 6 degrees below the horizon; at civil twilight, there is still enough light to navigate. Pytheas Simulator can also decide to leave at `'astronomical'` twilight, `'nautical'` twilight, or when the `'sun'` rises.

In [ ]:
travel = pytheas_simulator.travel.Travel(boat = hjortspring, map = travel_map, start_day = date_of_departure, 
                                         max_duration = max_duration_in_hours, timestep = timestep_in_minutes, 
                                         tolerance_distance = 1, twilights_of_stops='civil')

Now we are ready to run our simulation! Simply run

In [ ]:
travel.run(verbose=True)

Now the travel is finished, but we need to save the output to analyse it. Pytheas Simulator gives two types of outputs: a geojson output, containing all the data collected during the voyage, and an aggregate file containing a line of average quantities per voyage (this aggregate output is particularly useful for simulations over multiple dates). 
- The parameter `radius_for_success` determines within what radius (in km) from the landing site a voyage is considered successful. The reason you may want to change this is if you define a landing site which is on the coast: the data we use have square grids, and depending on the data quality, travels may hit land relatively far away from the landing site you use as input. This can be problematic in post-processing when you want to filter through successful voyages: by modifying this number, you can control how far from the landing site your boat can land to be still considered a successful trip.

In [ ]:
travel.output_geojson(f"./simulation_output/Outputs/{output}/{date_of_departure}-gothenburg-skagen") 
travel.append_to_aggregates(f"./simulation_output/Aggregates/{output}_gothenburg-skagen.csv", radius_for_success=10)

Finally, Pytheas Simulator comes with one function to visualize the travel. This function is part of the `Boat` module. 

In [ ]:
hjortspring.plot_trajectory(bounding_box)

### Coastal voyage

The coastal voyage works similarly to a crossing, except that we need to provide a route to the system for it to follow the coast. This is implemented in the module `pytheas_simulator.search`. In addition to adding a route, coastal travels can also stop during the night. We go again through the steps:

In [ ]:
max_duration_in_hours = 240 
date_of_departure = "1996-07-14"
final_day_of_navigation = "1996-07-24"
departure_site = locations["GOTHENBURG"]
landing_site = locations["TANUM"]
bounding_box = pytheas_simulator.utilities.bounding_box_from_points(departure_site, landing_site, buffer=4)
timestep_in_minutes = 15

travel_map = pytheas_simulator.map.Map(bounding_box, earliest_time=date_of_departure, latest_time=final_day_of_navigation, 
                                       wind_data_path = WIND_DATA, current_data_path=CURRENT_DATA, waves_data_path=WAVES_DATA)
departure_site_on_water = travel_map.find_closest_water(departure_site)

Now we define the route. The following snippet calculates a route between the `departure_site` and the `landing_site`, and saves it in the folder `./tutorial/routes`. The reason for saving it is that the routing is the single most expensive bit of Pytheas, since under the hood it implements an [A* search algorithm](https://en.wikipedia.org/wiki/A*_search_algorithm). By saving it, you can speed up any successive voyage by feeding directly the route to the algorithm. For the routing, land is considered as an impassable wall, thus the beginning and end of the route need to be fully on water. Not only, it's not impossible that the algorithm cannot find a route -- especially for very long routes that need to pass jagged environmental data. In such cases, play around with the parameters or do not hesitate to split the voyage in two legs.
- `target_interval`: this parameter controls how close together the targets need to be. If they are too close, the boat will be forced to go through lots of control points; if they are not close enough, there is a strong change for the boat to strive from the preferred route and aim towards obstacles.
- the algorithm extends the initial map 3 times (thus creating 4 layers): the first value refers to the most outwards part of the layer; the last value refers to land; the two parameters that deal with this determine which weights to assign to each layer, and how "thick" those layers will be:
    - `weights`: the land has a high weight (1000), as does the second to last layer (the second element). This creates a corridor of low weight passages close enought to the coast (the third value of the vector).
    - `iterations`: refers to the size of each layer. You can play around with these, but I found the current set-up to work well. 

In [ ]:
landing_site_on_water = travel_map.find_closest_water(landing_site)
route_GOT_TAN = travel_map.create_route(departure_site_on_water, landing_site, 
                                        target_interval = 2, weights = [1, 1000, 1, 1000], iterations = [10, 5, 3, 1])

route = pd.DataFrame(route_GOT_TAN)
route.to_csv(f'./routes/gothenburg_tanum.csv', index = False, header=False)

Note that from the moment the route is set-up, one can also import it directly from where it is saved. 

We set up the boat, then make it travel. We also define the travel so that it stops navigating at night (the limit of what is considered a night is controlled by `twilights_of_stops`). 

In [ ]:

hjortspring = pytheas_simulator.boat.Boat('Hjortspring', latitude = departure_site_on_water[0], longitude = departure_site_on_water[1],
                                          target = landing_site, speed_polar_diagram = speed_polar_diagram, 
                                          leeway_polar_diagram=leeway_polar_diagram, route_to_take = route_GOT_TAN)
travel = pytheas_simulator.travel.Travel(boat = hjortspring, map = travel_map, start_day = date_of_departure, 
                                         max_duration = max_duration_in_hours, timestep = timestep_in_minutes, 
                                         tolerance_distance = 2, twilights_of_stops='civil')

And finally, we run the model, save the output and visualize the trajectory:

In [ ]:
travel.output_geojson(f"./simulation_output/Outputs/{output}/{date_of_departure}_gothenburg-tanum") 
travel.append_to_aggregates(f"./simulation_output/Aggregates/{output}_gothenburg-tanum.csv", radius_for_success=10)
hjortspring.plot_trajectory(bounding_box)

## Conclusion

Now you know the ropes of Pytheas Simulator! As a next step, you can look up the script `./simulation.py` to see how a larger simulation is set up. In general, the two bottlenecks of the program are the creation of routes (and I showed in this tutorial how to avoid it), and the download of data for the `Map` object. This second one is unavoidable, but it can be lessened, by smartly cleaning the memory regularly along the simulation, and by simulating manageable chunks -- e.g. creating a new map object every month instead of every year. 

I hope this tutorial was useful for you! Do let me know if you find any errors or if this tutorial does not run for you. I will do my best to fix it and make it available to everybody. And if you feel that you could contribute, reach out!